# 01 – Load and clean the AFAD earthquake catalog

**Question:** How long does aftershock activity in Turkey take to return to its background level
after a large earthquake, and can the daily number of events be forecast from recent activity?

**Source:** AFAD earthquake catalog (deprem.afad.gov.tr), Turkey and surrounding region,
2010–2025, M ≥ 2.5. Used with attribution, not redistributed (see README).

**One row =** one earthquake event.

**Target:** daily number of events (`n_events`), built later in this notebook by counting events per day.

In [1]:
import pandas as pd

# Path is relative to the notebooks/ folder, so ../ goes up to the repo root first.
# The raw file isn't in the repo, see README for how to download it.
df = pd.read_csv("../data/raw/afad_turkey_2010-2025_m2.5.csv")

In [2]:
df.shape

(92986, 9)

In [3]:
df.dtypes

Date             str
Longitude    float64
Latitude     float64
Depth        float64
Rms          float64
Type             str
Magnitude    float64
Location         str
EventID        int64
dtype: object

In [4]:
df.head()

,Date,Longitude,Latitude,Depth,Rms,Type,Magnitude,Location,EventID
0,28/02/2025 15:12:30,25.79564,36.74816,6.63,0.52,ML,2.6,Ege Denizi,654019
1,28/02/2025 14:09:19,26.80487,38.84953,14.31,0.44,MW,4.0,Ege Denizi - [11.17 km] Dikili (İzmir),654005
2,28/02/2025 14:04:01,26.52097,36.58481,8.36,0.99,ML,2.5,Ege Denizi - [79.75 km] Bodrum (Muğla),654007
3,28/02/2025 10:45:34,36.13541,36.26677,8.71,0.42,ML,3.0,Antakya (Hatay),653995
4,28/02/2025 09:46:20,36.06519,39.06610,7.00,0.25,ML,2.6,Sarıoğlan (Kayseri),653991


In [5]:
df.tail()

,Date,Longitude,Latitude,Depth,Rms,Type,Magnitude,Location,EventID
92981,01/01/2010 02:20:06,26.0587,38.4995,6.77,0.6400,Md,2.7,"Ege Denizi, Sakız Adası (Hiyos) - [25.20 km] Ç...",96947
92982,01/01/2010 02:12:57,31.0495,39.0807,6.97,0.5400,Md,2.5,Emirdağ (Afyonkarahisar),96946
92983,01/01/2010 01:56:15,37.4918,38.3910,6.72,0.3100,Md,2.5,Darende (Malatya),96945
92984,01/01/2010 01:28:03,28.6497,35.5228,16.09,0.1242,Md,3.3,Akdeniz - [103.13 km] Kaş (Antalya),96944
92985,01/01/2010 00:53:44,31.2620,38.4587,7.11,0.2200,Md,2.7,Sultandağı (Afyonkarahisar),96943


In [6]:
#From the df.tail() and df.head(), it looks like there are different magnitude types, check how many there are
df["Type"].value_counts()

Type
ML     30327
Md     28614
Ml     28600
Mw      3709
MW      1735
Mwp        1
Name: count, dtype: int64

In [7]:
df.describe()

,Longitude,Latitude,Depth,Rms,Magnitude,EventID
count,92986.000000,92986.000000,92986.000000,92986.000000,92986.000000,92986.000000
mean,34.066880,38.457086,9.116436,0.471426,2.875820,274229.906889
std,6.149610,1.309886,7.662943,0.308350,0.412676,179980.136138
min,25.500000,35.500100,0.000000,0.000000,2.500000,96943.000000
25%,28.140650,37.490300,6.980000,0.300000,2.600000,132290.250000
50%,34.539750,38.578200,7.000000,0.440000,2.700000,175878.500000
75%,38.940442,39.166153,7.470000,0.610000,3.000000,453051.750000
max,45.000000,42.499800,191.300000,29.390000,7.700000,673342.000000


In [8]:
# Depth looks like it has a default value (median is exactly 7.0), so count how often it appears
n_default = (df["Depth"] == 7.0).sum()
print(n_default,"events have exactly 7 km depth, which is", round(n_default/len(df)*100,1),"% of all events")

21266 events have exactly 7 km depth, which is 22.9 % of all events


In [9]:
#check if there is any NaN values
df.isna().sum()

Date         0
Longitude    0
Latitude     0
Depth        0
Rms          0
Type         0
Magnitude    0
Location     0
EventID      0
dtype: int64

## Four things I notice

1. **Dates are stored as text** (`dtype: object`), in day/month/year format, and the file is
   sorted newest first. I need to parse them and sort oldest first before counting events per day.
2. **Location has several formats.** Most rows are "District (Province)", but offshore and
   cross-border events look like "Ege Denizi - [79.75 km] Bodrum (Muğla)" (visible in head() above)
3. **Many depths are exactly 7.0 km** (21,266 rows, about 23%). That looks like a default
   value rather than a real measurement.
4. **Magnitude types are mixed and inconsistently labelled.** `Type` contains ML, Ml, Md, Mw, MW
   and one Mwp. Should be checked if these types need to be converted to one single type, or are they the same unit?

In [10]:
#Dates are in text day/month/year format, tell pandas the exact format
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y %H:%M:%S")
df["Date"].head()

0   2025-02-28 15:12:30
1   2025-02-28 14:09:19
2   2025-02-28 14:04:01
3   2025-02-28 10:45:34
4   2025-02-28 09:46:20
Name: Date, dtype: datetime64[us]

In [11]:
#Sort oldest first
df = df.sort_values("Date").reset_index(drop=True)
df["Date"].head()

0   2010-01-01 00:53:44
1   2010-01-01 01:28:03
2   2010-01-01 01:56:15
3   2010-01-01 02:12:57
4   2010-01-01 02:20:06
Name: Date, dtype: datetime64[us]

In [12]:
#Extract parts of the date for grouping and plotting
# "day" is the date with the time set to 00:00; used later to count events per day.
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["hour"] = df["Date"].dt.hour
df["day"] = df["Date"].dt.normalize()

In [13]:
df[["Date", "year", "month", "hour", "day"]].head()

,Date,year,month,hour,day
0,2010-01-01 00:53:44,2010,1,0,2010-01-01
1,2010-01-01 01:28:03,2010,1,1,2010-01-01
2,2010-01-01 01:56:15,2010,1,1,2010-01-01
3,2010-01-01 02:12:57,2010,1,2,2010-01-01
4,2010-01-01 02:20:06,2010,1,2,2010-01-01
